<a href="https://colab.research.google.com/github/JUST-ZARI/hotel-clv-prediction/blob/main/notebooks/01_harmonize_dataset1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Get the project files.
# Colab: clone the repo (development branch). Locally: run from the repository root.
import os, sys
if "google.colab" in sys.modules:
    !git clone -b development https://github.com/JUST-ZARI/hotel-clv-prediction.git
    %cd /content/hotel-clv-prediction
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: C:\Projects\hotel-clv-prediction-1


## Importing libraries and setting paths

In [2]:
import pandas as pd

RAW_PATH = "data/raw/hotel_bookings.csv"
OUT_PATH = "data/processed/dataset1_harmonized.csv"


#Load raw data

In [3]:
df1 = pd.read_csv(RAW_PATH)
print("Shape:", df1.shape)
df1.head(3)

Shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


# Month mapping

In [4]:
# Convert month names to numbers
MONTH_MAP = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}

# Column mapping

In [5]:
# Rename columns to the common schema

COLUMN_MAP = {
    "hotel": "hotel_type",
    "is_canceled": "is_canceled",
    "lead_time": "lead_time",
    "arrival_date_year": "arrival_year",
    "arrival_date_month": "arrival_month",
    "arrival_date_day_of_month": "arrival_day",
    "stays_in_weekend_nights": "stays_weekend_nights",
    "stays_in_week_nights": "stays_week_nights",
    "adults": "num_adults",
    "children": "num_children",
    "babies": "num_babies",
    "meal": "meal_plan",
    "country": "country",
    "market_segment": "market_segment",
    "distribution_channel": "distribution_channel",
    "is_repeated_guest": "is_repeated_guest",
    "previous_cancellations": "previous_cancellations",
    "previous_bookings_not_canceled": "previous_bookings_not_canceled",
    "reserved_room_type": "room_type_reserved",
    "assigned_room_type": "assigned_room_type",
    "booking_changes": "booking_changes",
    "deposit_type": "deposit_type",
    "agent": "agent_id",
    "company": "company_id",
    "days_in_waiting_list": "days_in_waiting_list",
    "customer_type": "customer_type",
    "adr": "adr",
    "required_car_parking_spaces": "required_car_parking_spaces",
    "total_of_special_requests": "total_special_requests",
    "reservation_status": "reservation_status",
    "reservation_status_date": "reservation_status_date"
}

In [6]:
#Rename columns according to COLUMN_MAP
df1 = df1.rename(columns=COLUMN_MAP)
df1 = df1.drop(columns=["arrival_date_week_number"])
# Display the remaining column names
df1.columns.tolist()

['hotel_type',
 'is_canceled',
 'lead_time',
 'arrival_year',
 'arrival_month',
 'arrival_day',
 'stays_weekend_nights',
 'stays_week_nights',
 'num_adults',
 'num_children',
 'num_babies',
 'meal_plan',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'room_type_reserved',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent_id',
 'company_id',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_special_requests',
 'reservation_status',
 'reservation_status_date']

# Fixing data types

In [7]:
# Convert month names to numbers
df1["arrival_month"] = df1["arrival_month"].map(MONTH_MAP).astype(int)
# Fix numeric columns
df1["num_children"] = df1["num_children"].fillna(0).astype(int)
df1["agent_id"] = df1["agent_id"].astype("Int64")     # nullable int, keeps NaN
df1["company_id"] = df1["company_id"].astype("Int64")
# Fix text and date columns
df1["country"] = df1["country"].astype("string")      # keep NaN as missing, not "nan"
df1["reservation_status_date"] = pd.to_datetime(df1["reservation_status_date"])
# Convert binary columns to integers
df1["is_canceled"] = df1["is_canceled"].astype(int)
df1["is_repeated_guest"] = df1["is_repeated_guest"].astype(int)

# To remember -(Identify where the records came from)
df1["source_dataset"] = "hotel_bookings_2015_2017"

df1.dtypes

hotel_type                                   str
is_canceled                                int64
lead_time                                  int64
arrival_year                               int64
arrival_month                              int64
arrival_day                                int64
stays_weekend_nights                       int64
stays_week_nights                          int64
num_adults                                 int64
num_children                               int64
num_babies                                 int64
meal_plan                                    str
country                                   string
market_segment                               str
distribution_channel                         str
is_repeated_guest                          int64
previous_cancellations                     int64
previous_bookings_not_canceled             int64
room_type_reserved                           str
assigned_room_type                           str
booking_changes     

# Check remaining nulls

In [8]:
# Check missing values

nulls = df1.isnull().sum()

print("Columns with missing values:")
print(nulls[nulls > 0])

Columns with missing values:
country          488
agent_id       16340
company_id    112593
dtype: int64


#Save harmonized dataset

In [9]:
OUT_PATH = "data/processed/dataset1_harmonized.csv"
df1.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")

Saved to data/processed/dataset1_harmonized.csv
